# 02 - Silver layer: clean + star schema

**Input:** bronze Delta table (raw JSON)  
**Output (Unity Catalog `depi.hospitals`):** `hospitals_bronze` (parsed), `dim_patient_silver`, `dim_department_silver`,
`dim_hospital_silver`, `dim_date_silver`, `fact_table_silver`

Cleaning rules (nothing is invented - bad values become `NULL` + a flag column):

| Problem (injected by the producer) | Rule |
|---|---|
| Age > 100 or < 0 | `age = NULL`, `age_is_valid = false` |
| Admission time in the future | replaced by `event_ingestion_time`, `admission_time_corrected = true` |
| Missing discharge time | stays `NULL`, `has_discharge_time = false` (length of stay = `NULL`) |
| Discharge before admission | impossible -> treated as missing (`NULL`) |
| Same `patient_id` received twice | keep only the latest event |

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# ---------- Config ----------
STORAGE_ACCOUNT = "storageaccount5326"
CONTAINER = "hospitals"
SCHEMA = "depi.hospitals"          # catalog.schema
DIM_DATE_START, DIM_DATE_END = "2025-01-01", "2030-12-31"

bronze_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/raw_data"

SECRET_SCOPE = "hospital-scope"   # Databricks secret scope (see README -> "Secrets")

def get_secret(name):
    """Read a secret from the Databricks secret scope.
    Fallback: a notebook widget (paste the value at runtime - clear it before committing the notebook)."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=name)
    except Exception:
        dbutils.widgets.text(name, "")
        value = dbutils.widgets.get(name)
        if not value:
            raise ValueError(f"Secret '{name}' not found in scope '{SECRET_SCOPE}' and widget '{name}' is empty.")
        return value

spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net", get_secret("storage-key"))

def save_table(df, name):
    """Full rebuild of a silver table. overwriteSchema lets the table change shape between versions."""
    (df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.{name}"))

## 1) Parse the raw JSON (typed, but not cleaned yet)

In [ ]:
schema = """
    patient_id STRING,
    gender STRING,
    age INT,
    department STRING,
    admission_time STRING,
    discharge_time STRING,
    bed_id INT,
    hospital_id INT
"""

raw = spark.read.format("delta").load(bronze_path)
if "kafka_timestamp" not in raw.columns:           # rows written by the old bronze notebook
    raw = raw.withColumn("kafka_timestamp", F.lit(None).cast("timestamp"))

df_parsed = (
    raw
    .select(F.from_json(F.col("json_value"), schema).alias("data"), "kafka_timestamp")
    .select("data.*", "kafka_timestamp")
    .withColumn("admission_time", F.to_timestamp("admission_time"))
    .withColumn("discharge_time", F.to_timestamp("discharge_time"))
    .filter(F.col("patient_id").isNotNull())        # drops malformed JSON
)

save_table(df_parsed, "hospitals_bronze")

In [ ]:
df_parsed.printSchema()
display(df_parsed.limit(10))

## 2) Clean

In [ ]:
# Keep only the latest event per patient_id
latest_first = Window.partitionBy("patient_id").orderBy(F.col("kafka_timestamp").desc_nulls_last())

df_clean = (
    spark.table(f"{SCHEMA}.hospitals_bronze")
    .withColumn("_rn", F.row_number().over(latest_first))
    .filter("_rn = 1")
    .drop("_rn")

    # When the event reached Event Hub (fallback only for legacy rows without kafka_timestamp)
    .withColumn("event_ingestion_time", F.coalesce("kafka_timestamp", F.current_timestamp()))
    .drop("kafka_timestamp")

    # Age must be between 0 and 100, otherwise unknown (NULL) - no random values
    .withColumn("age_is_valid", F.coalesce(F.col("age").between(0, 100), F.lit(False)))
    .withColumn("age", F.when(F.col("age_is_valid"), F.col("age")))

    # A patient cannot be admitted after the event was received
    .withColumn("admission_time_corrected", F.col("admission_time") > F.col("event_ingestion_time"))
    .withColumn(
        "admission_time",
        F.when(F.col("admission_time_corrected"), F.col("event_ingestion_time")).otherwise(F.col("admission_time")),
    )

    # Discharge before admission is impossible -> unknown (NULL). Missing discharge stays NULL.
    .withColumn("discharge_time", F.when(F.col("discharge_time") >= F.col("admission_time"), F.col("discharge_time")))
    .withColumn("has_discharge_time", F.col("discharge_time").isNotNull())
)

In [ ]:
# Data-quality report: how many rows needed fixing?
(df_clean.agg(
    F.count("*").alias("rows"),
    F.sum((~F.col("age_is_valid")).cast("int")).alias("invalid_age"),
    F.sum(F.col("admission_time_corrected").cast("int")).alias("admission_in_future"),
    F.sum((~F.col("has_discharge_time")).cast("int")).alias("missing_or_invalid_discharge"),
).show())

## 3) Dimensions (one row per member - never per event)

In [ ]:
dim_patient = (
    df_clean.select("patient_id", "gender", "age", "age_is_valid")
    .withColumn(
        "age_group",
        F.when(F.col("age").isNull(), "Unknown")
         .when(F.col("age") < 18, "0-17")
         .when(F.col("age") < 40, "18-39")
         .when(F.col("age") < 60, "40-59")
         .otherwise("60+"),
    )
    # deterministic surrogate key: same patient_id -> same key on every run
    .withColumn("patient_sk", F.xxhash64("patient_id"))
)
save_table(dim_patient, "dim_patient_silver")

In [ ]:
dim_department = (
    df_clean.select("department").distinct()
    .withColumn("department_sk", F.row_number().over(Window.orderBy("department")).cast("int"))
)
save_table(dim_department, "dim_department_silver")
display(dim_department.orderBy("department_sk"))

In [ ]:
dim_hospital = (
    df_clean.select("hospital_id").distinct()
    .withColumn("hospital_name", F.concat(F.lit("Hospital Num "), F.col("hospital_id")))
)
save_table(dim_hospital, "dim_hospital_silver")

In [ ]:
dim_date = (
    spark.createDataFrame([(DIM_DATE_START, DIM_DATE_END)], ["start", "end"])
    .select(F.explode(F.sequence(F.col("start").cast("date"), F.col("end").cast("date"))).alias("date"))
    .withColumn("date_sk", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("year", F.year("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
)
save_table(dim_date, "dim_date_silver")

## 4) Fact table - grain: one row per patient admission event

In [ ]:
fact_table = (
    df_clean
    .withColumn("patient_sk", F.xxhash64("patient_id"))
    .join(F.broadcast(dim_department), on="department", how="left")      # 1 row per department -> no row multiplication
    .withColumn("admission_date_sk", F.date_format("admission_time", "yyyyMMdd").cast("int"))
    .withColumn("discharge_date_sk", F.date_format("discharge_time", "yyyyMMdd").cast("int"))
    .withColumn(
        "length_of_stay_hours",
        F.round((F.unix_timestamp("discharge_time") - F.unix_timestamp("admission_time")) / 3600, 2),
    )
    .select(
        "patient_sk", "department_sk", "hospital_id", "bed_id",
        "admission_time", "discharge_time",
        "admission_date_sk", "discharge_date_sk",
        "length_of_stay_hours", "has_discharge_time", "event_ingestion_time",
    )
)

In [ ]:
# ---------- Integrity checks (fail loudly instead of silently producing wrong KPIs) ----------
n_clean = df_clean.count()
n_fact = fact_table.count()
assert n_fact == n_clean, f"Fact has {n_fact} rows but clean data has {n_clean} -> a join is duplicating/dropping rows"

assert fact_table.filter("department_sk IS NULL").count() == 0, "Some facts have no matching department"

missing_dates = fact_table.join(dim_date, fact_table.admission_date_sk == dim_date.date_sk, "left_anti").count()
assert missing_dates == 0, f"{missing_dates} admissions fall outside dim_date ({DIM_DATE_START} -> {DIM_DATE_END})"

assert fact_table.filter("length_of_stay_hours < 0").count() == 0, "Negative length of stay found"

print(f"All checks passed - {n_fact} fact rows")

In [ ]:
save_table(fact_table, "fact_table_silver")
display(spark.table(f"{SCHEMA}.fact_table_silver").limit(20))